# Joins, nested aggregates, holistic aggregates, and library UDFs

The first two notebooks trace a single group-by. The data for sections 2 to 4 comes from the benchmark generator in `benchmarks/disc`, and every query is written out in the notebook. Real pipelines join tables, aggregate in several stages, use aggregates such as percentiles that need every value, and call compiled libraries inside their UDFs. This notebook shows Prism on each of those shapes.

In [1]:
import os
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from prism import Prism, udf

# one executor per worker, sized to the worker
spark = (SparkSession.builder.appName("prism-beyond")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.executor.cores", os.environ.get("WORKER_CORES", "2"))
         .config("spark.executor.memory", os.environ.get("WORKER_MEMORY", "2g"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
DATA = os.environ.get("PRISM_DATA", "/data")
SCALE = float(os.environ.get("PRISM_SCALE", "1"))   # multiplies every generated table's rows
print("Spark", spark.version, "on", spark.sparkContext.master, "| data scale", SCALE)

Spark 4.1.2 on spark://spark-master:7077 | data scale 1.0


## 1. A star join

Sales join a store table, a UDF computes each sale's net revenue, and the revenue is summed per region. One sale carries a discount of minus 500, a data-entry error that adds 500 to its net revenue. The trace walks back through the aggregate and then through the join, where each input of the join is a separate branch.

In [2]:
(spark.range(0, int(20_000 * SCALE), numPartitions=8)
 .select(F.col("id").alias("sale_id"), (F.col("id") % 50).cast("int").alias("store_id"),
         (1 + F.col("id") % 7).cast("double").alias("price"),
         (1 + F.col("id") % 3).cast("int").alias("qty"),
         F.when(F.col("id") == 12_345, F.lit(-500.0)).otherwise(F.lit(0.5)).alias("discount"),
         F.lit("paid in full").alias("memo"))
 .write.mode("overwrite").parquet(f"{DATA}/sales"))
(spark.range(0, 50)
 .select(F.col("id").cast("int").alias("store_id"),
         F.element_at(F.array(*[F.lit(r) for r in ["east", "west", "north", "south", "central"]]),
                      (F.col("id") % 5 + 1).cast("int")).alias("region"),
         F.lit("on site").alias("manager"))
 .write.mode("overwrite").parquet(f"{DATA}/stores"))

@udf(DoubleType())
def net(price, qty, discount):
    return float(price * qty - discount)

p = Prism(spark)
p.enable_capture()
p.enable_udf_aware(net, mode="lazy-cell")
p.enable_influence()
sales, stores = spark.read.parquet(f"{DATA}/sales"), spark.read.parquet(f"{DATA}/stores")
result = (sales.withColumn("net", net("price", "qty", "discount"))
          .join(stores, "store_id").groupBy("region").agg(F.sum("net").alias("revenue")))
rows = p.collect_with_lineage(result)
for row, _ in rows:
    print(row)

Row(region='east', revenue=30502.5)
Row(region='north', revenue=30000.0)
Row(region='west', revenue=30012.0)
Row(region='central', revenue=29988.0)
Row(region='south', revenue=29989.0)


In [3]:
row, out_id = max(rows, key=lambda r: r[0]["revenue"])
join_level = p.trace(result, [out_id]).go_back()
for branch, name in [(0, "sales"), (1, "stores")]:
    c = join_level.go_back(branch)
    while not c.at_scan:
        c = c.go_back()
    w = c.show()
    print(f"{name}: {len(w)} records traced, first: {w[0]}")
top_id, top_value = p.trace(result, [out_id]).influence("revenue").top(1)[0]
print(f"most influential sale: record {top_id}, influence {top_value}")
assert top_value > 500

sales: 4000 records traced, first: {'store_id': 0, 'price': 5.0, 'qty': 2, 'discount': 0.5, '_data': ['discount', 'price', 'qty'], '_control': [], '_regime': 'model'}


stores: 10 records traced, first: {'store_id': 15, 'region': 'east'}


most influential sale: record 2345, influence 505.0


## 2. Nested aggregation

Daily sales are summed per store and day, and the daily totals are averaged per store. One store recorded a day of 20 refunds as -1000 each, a sign error that drives its average below every other store's. Influence can be requested at either level by naming that level's output column. On the average it ranks the store's days, and on the daily total it ranks the sales. Both rankings order records by the size of their effect, whatever its direction.

In [4]:
import gen   # the benchmark data generator, used here only to write the tables
gen.gen_one(spark, "sales", n=int(300_000 * SCALE), out=f"{DATA}/disc")

@udf(DoubleType())
def amount(raw):
    return float(raw)                             # a sale's amount, where a refund is negative

p = Prism(spark)
p.enable_capture()                                # record which sales feed each output
p.enable_udf_aware(amount)                        # record which fields amount read
p.enable_influence()                              # record contributions at both aggregate levels

sales = spark.read.parquet(f"{DATA}/disc/sales.parquet")
daily = (sales.withColumn("a", amount("raw"))
         .groupBy("store", "day").agg(F.sum("a").alias("daily")))       # level 1: total per store and day
result = daily.groupBy("store").agg(F.avg("daily").alias("avg_daily"))  # level 2: average daily total per store
rows = p.collect_with_lineage(result)
worst_row, worst_id = min(rows, key=lambda r: r[0]["avg_daily"])        # the store with the lowest average
bad = [worst_id]
print("suspicious output:", worst_row)
days = sorted(p.trace(result, bad).influence("avg_daily").scores.values(), key=abs, reverse=True)
sales = sorted(p.trace(result, bad).influence("daily").scores.values(), key=abs, reverse=True)
print("day influence on the store average:", [round(v, 1) for v in days[:3]], "...")
print("sale influence on its daily total: ", sales[:3], "...", "refunds ranked first:",
      sales[:20].count(-1000.0))
assert abs(days[0]) > 5 * abs(days[1]) and sales[:20].count(-1000.0) == 20

wrote ~300000 sales rows (FG=300 faulty group) -> /data/disc/sales.parquet


suspicious output: Row(store=99, avg_daily=-1860.8)


day influence on the store average: [-2009.8, 224.4, 224.1] ...
sale influence on its daily total:  [-1000.0, -1000.0, -1000.0] ... refunds ranked first: 20


## 3. A holistic aggregate

A percentile has no constant-size summary, so Spark computes it with an object-hash aggregate and Prism falls back to leave-one-out influence, the change in the percentile when each reading is removed. In this table 15 of the store's 300 readings are 1000, exactly the top 5%. Removing one of them lowers the 95th percentile, and removing any other reading raises it by the same amount. The sign of the influence separates the two groups.

In [5]:
gen.gen_one(spark, "latency", n=int(300_000 * SCALE), out=f"{DATA}/disc")

@udf(DoubleType())
def latency_ms(raw):
    return float(raw)                             # one request's latency in milliseconds

p = Prism(spark)
p.enable_capture()
p.enable_udf_aware(latency_ms)
p.enable_influence()                              # leave-one-out influence for the percentile

readings = spark.read.parquet(f"{DATA}/disc/latency.parquet")
result = (readings.withColumn("ms", latency_ms("raw"))
          .groupBy("store").agg(F.percentile("ms", F.lit(0.95)).alias("p95")))  # exact 95th percentile per store
rows = p.collect_with_lineage(result)
worst_row, worst_id = max(rows, key=lambda r: r[0]["p95"])             # the store with the highest p95
bad = [worst_id]
print("suspicious output:", worst_row)
scores = p.trace(result, bad).influence("p95").scores.values()
raised = sorted(v for v in scores if v > 0)
print(f"{len(raised)} readings lowered the p95 when removed, by {raised[0]:.2f} each")
assert len(raised) == 15

wrote ~300000 latency rows (FG=300 faulty group) -> /data/disc/latency.parquet


suspicious output: Row(store=99, p95=58.550000000011266)


15 readings lowered the p95 when removed, by 49.55 each


## 4. A UDF that calls numpy

This is the weather benchmark's unit conversion written with numpy. A taint-tracking wrapper cannot enter `np.multiply`, because compiled code needs a real number rather than a wrapper object. Prism keeps values plain and attributes each library call's result to its arguments, so the field lineage is the same as for plain Python.

In [6]:
import numpy as np
gen.gen_one(spark, "weather", n=int(300_000 * SCALE), out=f"{DATA}/disc")

@udf(DoubleType())
def to_mm_np(value, unit):
    v = np.float64(value)                                       # the reading as a numpy scalar
    return float(v if unit == "mm" else np.multiply(v, 304.8))  # same bug, computed in compiled numpy code

p = Prism(spark)
p.enable_capture()
p.enable_udf_aware(to_mm_np, mode="lazy-cell")    # field lineage through the numpy calls
readings = spark.read.parquet(f"{DATA}/disc/weather.parquet")
result = (readings.withColumn("mm", to_mm_np("value", "unit"))
          .groupBy("month").agg(F.max("mm").alias("max_mm")))
rows = p.collect_with_lineage(result)
bad = [max(rows, key=lambda r: r[0]["max_mm"])[1]]  # the month with the implausible maximum
cur = p.trace(result, bad)
while not cur.at_scan:
    cur = cur.go_back()
faulty = [w for w in cur.show() if w["unit"] == "in"][0]
print(faulty)
assert faulty["_data"] == ["value"] and faulty["_control"] == ["unit"]

wrote ~300000 weather rows (FG=300 faulty group) -> /data/disc/weather.parquet


{'value': 90.0, 'unit': 'in', 'month': 12, '_data': ['value'], '_control': ['unit'], '_regime': 'model'}


In [7]:
spark.stop()